# Wholesale Customer Spending EDA
Basic Exploratory Data Analysis (no machine learning) on the UCI Wholesale Customers dataset (440 rows).

Dataset: https://archive.ics.uci.edu/dataset/292/wholesale+customers

# 1. Import Libraries and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the dataset directly from the UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"

try:
    data = pd.read_csv(url)
except Exception:
    # Fallback: official UCI python package (dataset id = 292)
    # !pip install ucimlrepo
    from ucimlrepo import fetch_ucirepo
    wholesale = fetch_ucirepo(id=292)
    data = pd.concat([wholesale.data.features, wholesale.data.targets], axis=1)

print("Dataset loaded successfully")

# 2. Creating a Dataset

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df.head()

In [ ]:
df.tail()

# 3. Observing the Dataframe

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### 1. Checking for Null Values

In [ ]:
df.isnull().sum()

### 2. Checking for Duplicate Values

In [ ]:
df.duplicated().sum()

# 4. Preparing the Data

Channel and Region are stored as numbers. As per the dataset description:

- Channel: 1 = Horeca (Hotel/Restaurant/Cafe), 2 = Retail
- Region: 1 = Lisbon, 2 = Oporto, 3 = Other

In [ ]:
df["Channel_Name"] = df["Channel"].map({1: "Horeca", 2: "Retail"})
df["Region_Name"] = df["Region"].map({1: "Lisbon", 2: "Oporto", 3: "Other"})

# The six spending categories
categories = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

df.head(2)

# 5. Question 1: Customers in Each Channel and Region

### 1. Customers per Channel

In [ ]:
channel_counts = df["Channel_Name"].value_counts()
channel_counts

### 2. Customers per Region

In [ ]:
region_counts = df["Region_Name"].value_counts()
region_counts

### 3. Customers per Channel and Region together

In [ ]:
df.groupby(["Channel_Name", "Region_Name"]).size()

# 6. Question 2: Highest Total and Average Spending

In [ ]:
# NumPy array of the six spending columns (440 rows x 6 columns)
spending = df[categories].to_numpy()
spending.shape

In [ ]:
# Total and average spending of each category using NumPy (axis=0 -> down the rows)
total_spending = np.sum(spending, axis=0)
average_spending = np.mean(spending, axis=0)

category_summary = pd.DataFrame({
    "Total": total_spending,
    "Average": average_spending.round(2)
}, index=categories)

category_summary.sort_values("Total", ascending=False)

In [ ]:
print("Highest total spending   :", category_summary["Total"].idxmax(), "->", category_summary["Total"].max())
print("Highest average spending :", category_summary["Average"].idxmax(), "->", category_summary["Average"].max())

# 7. Question 3: Mean, Median, Minimum and Maximum for Each Category

In [ ]:
stats = pd.DataFrame({
    "Mean":   np.mean(spending, axis=0).round(2),
    "Median": np.median(spending, axis=0),
    "Min":    np.min(spending, axis=0),
    "Max":    np.max(spending, axis=0)
}, index=categories)

stats

# 8. Question 4: Top 10 Customers by Total Spending

In [ ]:
# Total spending of each customer across all six categories
df["Total_Spending"] = df[categories].sum(axis=1)

top10 = df.sort_values("Total_Spending", ascending=False).head(10)
top10[["Channel_Name", "Region_Name"] + categories + ["Total_Spending"]]

# 9. Question 5: Average Category Spending by Channel

In [ ]:
channel_avg = df.groupby("Channel_Name")[categories].mean().round(2)
channel_avg

In [ ]:
# Transposed view for easier comparison (rows = categories)
channel_avg.T

# 10. Filtering Data

### 1. Customers who spend more than the average on Grocery

In [ ]:
high_grocery = df[df["Grocery"] > df["Grocery"].mean()]
print("Number of customers:", high_grocery.shape[0])
high_grocery["Channel_Name"].value_counts()

### 2. Retail customers from Lisbon

In [ ]:
retail_lisbon = df[(df["Channel_Name"] == "Retail") & (df["Region_Name"] == "Lisbon")]
print("Number of customers:", retail_lisbon.shape[0])
retail_lisbon.head()

# 11. Data Visualisation

### 1. Bar Chart: Number of Customers per Channel

In [ ]:
plt.bar(channel_counts.index, channel_counts.values, color=["steelblue", "orange"])

plt.xlabel("Channel")
plt.ylabel("Number of Customers")
plt.title("Customers per Channel")

plt.show()

### 2. Bar Chart: Total Spending per Category

In [ ]:
plt.bar(categories, total_spending, color="seagreen")

plt.xlabel("Category")
plt.ylabel("Total Annual Spending")
plt.title("Total Spending per Category")
plt.xticks(rotation=45)

plt.show()

# 12. Observations

1. **Channel and region:** Most customers belong to the Horeca channel (298 customers) compared to Retail (142 customers), and the largest share of customers comes from the `Other` region (316 customers), far ahead of Lisbon and Oporto.
2. **Spending categories:** Fresh has the highest total and average spending, followed by Grocery and Milk. Delicassen has the lowest spending. The max values are much larger than the medians, so a few very large customers pull the averages up.
3. **Channel comparison:** Retail customers spend more on average on Milk, Grocery and Detergents_Paper, while Horeca customers spend more on Fresh and Frozen products. The data also has no missing values and no duplicate rows.